In [3]:
import sqlite3
conn = sqlite3.connect("guardian_rankings.db")
for row in conn.execute("SELECT sql FROM sqlite_master WHERE type='table'"):
    print(row[0])
    print()
conn.close()

CREATE TABLE "guardian_ranking" (
"ranking_year" INTEGER,
  "ranking" INTEGER,
  "ranking_prev" INTEGER,
  "ranking_change" INTEGER,
  "institution" TEXT,
  "guardian_score_per_100" REAL,
  "nss_teaching_pct" REAL,
  "nss_overall_pct" REAL,
  "expenditure_per_student_per_10" REAL,
  "student_staff_ratio" REAL,
  "career_prospects_pct" REAL,
  "value_added_score_per_10" REAL,
  "entry_tariff" REAL,
  "nss_feedback_pct" REAL
)

CREATE TABLE "guardian_subject_areas" ("subject_area_year" INTEGER, "subject_area" TEXT, "subject_area_rank" INTEGER, "subject_area_rank_prev" INTEGER, "subject_area_rank_change" INTEGER, "institution" TEXT, "guardian_score_per_100" REAL, "pct_satisfied_with_teaching" REAL, "pct_satisfied_overall_with_course" REAL, "expenditure_per_student_fte" REAL, "student_staff_ratio" REAL, "career_prospects" REAL, "value_added_score_per_10" REAL, "average_entry_tariff" REAL, "pct_satisfied_with_assessment" REAL)



## Testy v rámci přípravy dat do relační podoby

### Test unikátnosti PK u tabulky guarding_ranking

In [14]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("guardian_rankings.db")

df = pd.read_sql_query("""
    SELECT ranking_year, institution, COUNT(*) as cnt
    FROM guardian_ranking
    GROUP BY ranking_year, institution
    HAVING cnt > 1
""", conn)

print(df.shape)
print(df.head(10))

(0, 3)
Empty DataFrame
Columns: [ranking_year, institution, cnt]
Index: []


### Test unikátnosti PK u tabulky guarding_subject_areas

In [17]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("guardian_rankings.db")

df = pd.read_sql_query("""
    SELECT subject_area_year, subject_area, institution, COUNT(*) as cnt
    FROM guardian_subject_areas
    GROUP BY subject_area_year, subject_area, institution
    HAVING cnt > 1
""", conn)

print(df.shape)
print(df.head(10))

(0, 4)
Empty DataFrame
Columns: [subject_area_year, subject_area, institution, cnt]
Index: []


## Test převod rank sloupců na INTEGER

`subject_area_rank_prev` a `subject_area_rank_change` mají v databázi typ REAL, protože kvůli NULL hodnotám byly při importu z Excelu načteny jako float64. Hodnoty jsou ale vždy celočíselné (ověřeno výše), takže sloupce přetypujeme na INTEGER.

In [2]:
import sqlite3

conn = sqlite3.connect("guardian_rankings.db")
cur = conn.cursor()

# Sloupce, které mají v názvu "rank", ale v DB nemají affinity INTEGER
rank_columns_to_fix = {
    "guardian_subject_areas": ["subject_area_rank_prev", "subject_area_rank_change", "subject_area_rank_prev", "subject_area_rank_change"],
}

for table, columns in rank_columns_to_fix.items():
    # Zjistíme aktuální sloupce tabulky a definice jejích indexů, abychom je po přestavbě obnovili
    col_info = cur.execute(f"PRAGMA table_info({table})").fetchall()
    index_defs = cur.execute(
        "SELECT sql FROM sqlite_master WHERE type='index' AND tbl_name=? AND sql IS NOT NULL",
        (table,),
    ).fetchall()

    # Nová definice tabulky - u zadaných sloupců změníme deklarovaný typ na INTEGER
    col_names = [row[1] for row in col_info]
    col_defs = [
        f'"{name}" {"INTEGER" if name in columns else col_type}'
        for _, name, col_type, *_ in col_info
    ]

    tmp_table = f"{table}__tmp"
    cur.execute(f'CREATE TABLE "{tmp_table}" ({", ".join(col_defs)})')

    select_cols = ", ".join(
        f'CAST("{name}" AS INTEGER)' if name in columns else f'"{name}"'
        for name in col_names
    )
    cur.execute(f'INSERT INTO "{tmp_table}" SELECT {select_cols} FROM "{table}"')

    cur.execute(f'DROP TABLE "{table}"')
    cur.execute(f'ALTER TABLE "{tmp_table}" RENAME TO "{table}"')

    # Indexy zanikly spolu s původní tabulkou (DROP TABLE), takže je znovu vytvoříme
    for (index_sql,) in index_defs:
        cur.execute(index_sql)

    print(f"Opraveno: {table} -> {columns}")

conn.commit()

# Ověření výsledného schématu
for row in cur.execute("SELECT sql FROM sqlite_master WHERE type='table'"):
    print(row[0])
    print()

conn.close()

Opraveno: guardian_subject_areas -> ['subject_area_rank_prev', 'subject_area_rank_change', 'subject_area_rank_prev', 'subject_area_rank_change']
CREATE TABLE "guardian_ranking" (
"ranking_year" INTEGER,
  "ranking" INTEGER,
  "ranking_prev" INTEGER,
  "ranking_change" INTEGER,
  "institution" TEXT,
  "guardian_score_per_100" REAL,
  "nss_teaching_pct" REAL,
  "nss_overall_pct" REAL,
  "expenditure_per_student_per_10" REAL,
  "student_staff_ratio" REAL,
  "career_prospects_pct" REAL,
  "value_added_score_per_10" REAL,
  "entry_tariff" REAL,
  "nss_feedback_pct" REAL
)

CREATE TABLE "guardian_subject_areas" ("subject_area_year" INTEGER, "subject_area" TEXT, "subject_area_rank" INTEGER, "subject_area_rank_prev" INTEGER, "subject_area_rank_change" INTEGER, "institution" TEXT, "guardian_score_per_100" REAL, "pct_satisfied_with_teaching" REAL, "pct_satisfied_overall_with_course" REAL, "expenditure_per_student_fte" REAL, "student_staff_ratio" REAL, "career_prospects" REAL, "value_added_score_